In [ ]:
import pandas as pd, numpy as np, pickle
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report, confusion_matrix

df = pd.read_csv("merged_dataset.csv")

drop_cols = ["customer_id", "customer_id_new", "transaction_id",
             "purchase_date", "product_category"]
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].astype(float).values
le = LabelEncoder()
y = le.fit_transform(df["product_category"])

print("X:", X.shape, "| classes:", list(le.classes_))
print(df["product_category"].value_counts())

X: (117, 28) | classes: ['Books', 'Clothing', 'Electronics', 'Groceries', 'Sports']
product_category
Sports         28
Electronics    27
Clothing       22
Groceries      20
Books          20
Name: count, dtype: int64


In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

scaler = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler.transform(X_tr), scaler.transform(X_te)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42),
    "LogisticRegression": LogisticRegression(max_iter=2000, random_state=42),
}

results = {}
for name, m in models.items():
    m.fit(Xtr_s, y_tr)
    pred, proba = m.predict(Xte_s), m.predict_proba(Xte_s)
    results[name] = {
        "accuracy": accuracy_score(y_te, pred),
        "f1_macro": f1_score(y_te, pred, average="macro"),
        "log_loss": log_loss(y_te, proba, labels=np.arange(len(le.classes_))),
    }
    print(f"\n=== {name} ===")
    for k, v in results[name].items():
        print(f"  {k:10s}: {v:.4f}")
    print(classification_report(y_te, pred, target_names=le.classes_, zero_division=0))

pd.DataFrame(results).T


=== RandomForest ===
  accuracy  : 0.3000
  f1_macro  : 0.2963
  log_loss  : 1.8454
              precision    recall  f1-score   support

       Books       0.50      0.20      0.29         5
    Clothing       0.50      0.50      0.50         6
 Electronics       0.33      0.14      0.20         7
   Groceries       0.11      0.20      0.14         5
      Sports       0.30      0.43      0.35         7

    accuracy                           0.30        30
   macro avg       0.35      0.29      0.30        30
weighted avg       0.35      0.30      0.30        30


=== LogisticRegression ===
  accuracy  : 0.2000
  f1_macro  : 0.2019
  log_loss  : 2.7692
              precision    recall  f1-score   support

       Books       0.17      0.20      0.18         5
    Clothing       0.67      0.33      0.44         6
 Electronics       0.00      0.00      0.00         7
   Groceries       0.10      0.20      0.13         5
      Sports       0.22      0.29      0.25         7

    accur

,accuracy,f1_macro,log_loss
RandomForest,0.3,0.296303,1.845443
LogisticRegression,0.2,0.201919,2.769182


In [ ]:
best = models["RandomForest"]
imp = pd.Series(best.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(imp.head(12))

artifact = {"model": best, "scaler": scaler, "feature_cols": feature_cols,
            "label_encoder": le}
with open("product_model.pkl", "wb") as f:
    pickle.dump(artifact, f)
print("saved product_model.pkl")

customer_rating          0.074492
purchase_amount          0.074338
spend_vs_avg             0.070012
cust_total_spend         0.069761
spend_per_engagement     0.066641
cust_avg_spend           0.063353
interest_mean            0.060210
purchase_dow             0.055558
purchase_month           0.054026
engagement_x_interest    0.051433
interest_max             0.045096
interest_x_sentiment     0.041956
dtype: float64
saved product_model.pkl


In [ ]:
df_img = pd.read_csv("image_features.csv")

feat_cols = [c for c in df_img.columns if c.startswith("feat_")]
df_img = df_img.drop_duplicates(subset=feat_cols).reset_index(drop=True)

# group = the source image, so all its augmentations stay together
df_img["source"] = df_img["image_id"].str.replace(
    r"_(original|rotate|flip|grayscale|brightness)$", "", regex=True)

# hold out the "surprised" photo of each member as the test set
test_mask = df_img["expression"] == "surprised"
train_df, test_df = df_img[~test_mask], df_img[test_mask]

X_tr, y_tr = train_df[feat_cols].values, train_df["label"].values
X_te, y_te = test_df[feat_cols].values,  test_df["label"].values
print("train:", X_tr.shape, "test:", X_te.shape)
print("test members:", sorted(set(y_te)))

train: (32, 1280) test: (16, 1280)
test members: ['dan', 'enock', 'hugues', 'urlich']


In [ ]:
scaler = StandardScaler().fit(X_tr)
face_model = RandomForestClassifier(n_estimators=300, random_state=42)
face_model.fit(scaler.transform(X_tr), y_tr)

pred  = face_model.predict(scaler.transform(X_te))
proba = face_model.predict_proba(scaler.transform(X_te))

print("Accuracy:", accuracy_score(y_te, pred))
print("F1 (macro):", f1_score(y_te, pred, average="macro"))
print("Log loss:", log_loss(y_te, proba, labels=face_model.classes_))
print(classification_report(y_te, pred, zero_division=0))
print(pd.DataFrame(confusion_matrix(y_te, pred),
                   index=face_model.classes_, columns=face_model.classes_))

with open("face_model.pkl", "wb") as f:
    pickle.dump({"model": face_model, "scaler": scaler,
                 "feature_cols": feat_cols}, f)
print("saved face_model.pkl")

Accuracy: 1.0
F1 (macro): 1.0
Log loss: 0.7828741087930065
              precision    recall  f1-score   support

         dan       1.00      1.00      1.00         4
       enock       1.00      1.00      1.00         4
      hugues       1.00      1.00      1.00         4
      urlich       1.00      1.00      1.00         4

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16

        dan  enock  hugues  urlich
dan       4      0       0       0
enock     0      4       0       0
hugues    0      0       4       0
urlich    0      0       0       4
saved face_model.pkl


In [ ]:
df_aud = pd.read_csv("audio_features.csv")

aud_feats = [c for c in df_aud.columns
             if c.startswith("mfcc_") or c in ("spectral_rolloff", "energy")]

# train on one phrase, test on the other -> text-independent verification
train_df = df_aud[df_aud["phrase"] == "yes_approve"]
test_df  = df_aud[df_aud["phrase"] == "confirm_transaction"]

X_tr, y_tr = train_df[aud_feats].values, train_df["label"].values
X_te, y_te = test_df[aud_feats].values,  test_df["label"].values
print("train:", X_tr.shape, "test:", X_te.shape)

train: (16, 15) test: (16, 15)


In [ ]:
scaler = StandardScaler().fit(X_tr)
voice_model = RandomForestClassifier(n_estimators=300, random_state=42)
voice_model.fit(scaler.transform(X_tr), y_tr)

pred  = voice_model.predict(scaler.transform(X_te))
proba = voice_model.predict_proba(scaler.transform(X_te))

print("Accuracy:", accuracy_score(y_te, pred))
print("F1 (macro):", f1_score(y_te, pred, average="macro"))
print("Log loss:", log_loss(y_te, proba, labels=voice_model.classes_))
print(classification_report(y_te, pred, zero_division=0))

with open("voice_model.pkl", "wb") as f:
    pickle.dump({"model": voice_model, "scaler": scaler,
                 "feature_cols": aud_feats}, f)
print("saved voice_model.pkl")

Accuracy: 0.875
F1 (macro): 0.8740079365079365
Log loss: 0.7367419193057437
              precision    recall  f1-score   support

         dan       0.75      0.75      0.75         4
       enock       0.80      1.00      0.89         4
      hugues       1.00      1.00      1.00         4
      urlich       1.00      0.75      0.86         4

    accuracy                           0.88        16
   macro avg       0.89      0.88      0.87        16
weighted avg       0.89      0.88      0.87        16

saved voice_model.pkl


In [ ]:
from google.colab import files
for f in ["product_model.pkl", "face_model.pkl", "voice_model.pkl"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>